# Surprise Potential Closed-Loop Sweep (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/surprise-potential/notebooks/surprise_potential_closedloop_colab.ipynb)

## Objective
Run a restart-safe, contract-aligned surprise-potential closed-loop sweep on Waymax with the UniMM-style rollout-belief backend.

### Inputs
- Metric list and paper counterfactual family list.
- Persistent Drive root.
- Sharding and resume policy.

### Outputs
- Per-metric sweep summaries.
- Exported artifact index for each run.
- Notebook-level sweep report under persistent storage.

### Success Criteria
- Fast-fail stage passes (probe + preflight + gate) before main execution.
- Main sweep runs only when enabled and fast-fail is ready.
- Artifacts/manifests persist under Drive-backed storage.


## Step 1 - Bootstrap Cell

Check runtime identity and basic prerequisites before storage/repo operations.


In [ ]:
import os
import platform
import sys

print('[bootstrap] python =', sys.version.split()[0])
print('[bootstrap] platform =', platform.platform())
print('[bootstrap] executable =', sys.executable)

IS_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
print('[bootstrap] is_colab =', IS_COLAB)
if not IS_COLAB:
    print('[bootstrap] warning: notebook is designed for Colab + Drive persistence.')


## Step 2 - Storage Cell

Mount Drive and verify the persistent folder is writable.


In [ ]:
import os
import time
from pathlib import Path

REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

if IS_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')

required_drive_path = Path(REQUIRED_DRIVE_FOLDER)
required_drive_path.mkdir(parents=True, exist_ok=True)
probe_path = required_drive_path / f'.write_probe_{int(time.time() * 1e6)}.tmp'
probe_path.write_text('ok', encoding='utf-8')
probe_path.unlink(missing_ok=True)
print('[storage] writable =', required_drive_path)


## Step 3 - Repo Sync Cell

Clone/pull repo, install missing dependencies via deterministic setup, and prepare imports.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for pth in (REPO_DIR, f'{REPO_DIR}/src'):
    if pth not in sys.path:
        sys.path.insert(0, pth)

from src.platform import (
    ensure_drive_ready,
    ensure_repo_checkout,
    prepare_repo_imports,
    run_cached_deterministic_setup,
)

repo_sync = ensure_repo_checkout(repo_url=REPO_URL, repo_dir=REPO_DIR, branch=REPO_BRANCH)
drive_status = ensure_drive_ready(
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    verify_drive_access_every_run=False,
)
prepared_repo_dir = prepare_repo_imports(repo_dir=repo_sync.repo_dir, force_module_hot_reload=True)
setup_result = run_cached_deterministic_setup(
    repo_root=prepared_repo_dir,
    force_reinstall=False,
    auto_restart_after_setup=True,
    strict_lockfile_check=True,
    setup_cache_enabled=True,
    revalidate_core_imports_on_cache_hit=True,
    setup_cache_path='/content/.surprise_potential_setup_cache.json',
    repo_rev=repo_sync.repo_rev,
)

print('[repo] dir =', prepared_repo_dir)
print('[repo] rev =', repo_sync.repo_rev)
print('[storage] drive_ready =', drive_status.writable)
print('[setup] cache_hit =', setup_result.cache_hit)

if bool(setup_result.result.get('restart_required', False)):
    raise SystemExit('[setup] runtime restart required. Re-run notebook from Step 1 after restart.')

from src.workflows import (
    build_surprise_potential_report_frames,
    build_surprise_potential_rigorous_eval,
    export_surprise_potential_report,
    initialize_run_context,
    load_surprise_potential_report,
    report_run_context,
    report_surprise_potential_sweep,
    run_surprise_potential_metric_sweep,
    run_surprise_potential_single_metric,
)


## Step 4 - Configuration Cell

Define all notebook controls in a single config object.


In [ ]:
EXPERIMENT = {
    # Mandatory contract fields
    'RUN_NAME': '',
    'RUN_PREFIX': 'surprise_potential',
    'PERSIST_ROOT': '/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1',
    'REPORT_PERSIST_ROOT': '/content/drive/MyDrive/waymax_experiments/surprise_potential_reports',
    'N_SHARDS': 5,
    'SHARD_ID': 'auto',
    'RESUME_FROM_EXISTING': True,
    'RUN_ENABLED': True,

    # Resume-aware orchestration
    'AUTO_LOAD_EXISTING_REPORT_IF_AVAILABLE': True,
    'FORCE_RERUN_MAIN_SWEEP': False,

    # Canonical run controls
    'RUN_TAG': '',
    'RUN_TAG_PREFIX': 'surprise_potential',
    'RUN_MODE': 'auto',  # auto | fresh | resume
    'AUTO_RUN_MAIN_LOOP_WHEN_READY': True,
    'RUN_MAIN_LOOP_OVERRIDE': None,

    # Method controls (keep all metrics, optionally reduce methods to save runtime)
    'METHODS': [
        'random',
        'risk_only',
        'surprise_only',
        'joint',
    ],

    # Experiment scope: full surprise-metric sweep
    'METRICS': [
        'unimm_rollout_kl',
        'predictive_seq_w2',
        'predictive_seq_kl',
        'predictive_w2',
        'predictive_kl',
        'action_kl',
        'latent_belief_kl',
    ],
    'COUNTERFACTUAL_FAMILIES': [
        'hist_prim',
    ],
    'PLANNER_BACKEND': 'unimm_style',
    'CONTINUE_ON_ERROR': True,
    'WARN_ON_CONFIG_DRIFT': True,
    'RESTORE_FROM_UPLOAD': False,

    # Main sweep sizing
    'N_EVAL_SCENARIOS': 100,
    'STRICT_MIN_EVAL': 100,
    'N_TOTAL_SCENARIOS_FLOOR': 400,

    # Fast-fail sizing (cheap validation pass)
    'FAST_FAIL_N_EVAL_SCENARIOS': 16,
    'FAST_FAIL_STRICT_MIN_EVAL': 16,
    'FAST_FAIL_N_TOTAL_SCENARIOS_FLOOR': 120,

    # Rigorous eval/export controls
    'RIGOROUS_BOOTSTRAP_SAMPLES': 1500,
    'RIGOROUS_RANDOM_SEED': 17,
    'REPORT_WRITE_PLOTS': True,

    'QUICK_PROBE_SETTINGS': {
        'quick_probe_scenarios': 1,
        'quick_probe_proposals_per_scenario': 4,
        'stop_if_quick_probe_collapsed': False,
        'auto_escalate_quick_probe': True,
        'max_probe_escalations': 3,
        'probe_scale_multipliers': (1.0, 1.35, 1.8),
        'probe_delta_l2_multipliers': (1.0, 1.2, 1.4),
        'probe_delta_clip_multipliers': (1.0, 1.1, 1.2),
        'probe_budget_bump_per_escalation': 2,
        'apply_successful_probe_tuning': True,
        'probe_belief_source_modes': ('b4', 'b3', 'auto'),
    },

    'SWEEP_KWARGS': {
        'quick_probe_enabled': True,
        'surprise_gate_enabled': True,
        'restore_from_upload': False,
        'stop_on_preflight_fail': False,
        'stop_on_gate_fail': False,
        'allow_main_loop_when_gate_fails': False,
        'warn_on_config_drift': True,
    },
}

print('[config] metrics =', EXPERIMENT['METRICS'])
print('[config] families =', EXPERIMENT['COUNTERFACTUAL_FAMILIES'])
print('[config] methods =', EXPERIMENT['METHODS'])
print('[config] persist_root =', EXPERIMENT['PERSIST_ROOT'])
print('[config] report_root =', EXPERIMENT['REPORT_PERSIST_ROOT'])


## Step 5 - Run Context Cell

Resolve run tag, resume mode, and shard routing before any expensive work.


In [ ]:
from pathlib import Path

run_context = initialize_run_context(
    run_tag=str(EXPERIMENT['RUN_TAG']),
    persist_root=str(EXPERIMENT['PERSIST_ROOT']),
    n_shards=int(EXPERIMENT['N_SHARDS']),
    shard_id=EXPERIMENT['SHARD_ID'],
    auto_run_main_loop_when_ready=bool(EXPERIMENT['AUTO_RUN_MAIN_LOOP_WHEN_READY']),
    run_main_loop_override=EXPERIMENT['RUN_MAIN_LOOP_OVERRIDE'],
    n_eval_scenarios=int(EXPERIMENT['N_EVAL_SCENARIOS']),
    strict_min_eval=int(EXPERIMENT['STRICT_MIN_EVAL']),
    n_total_scenarios_floor=int(EXPERIMENT['N_TOTAL_SCENARIOS_FLOOR']),
    run_tag_prefix=str(EXPERIMENT['RUN_TAG_PREFIX']),
    planner_backend=str(EXPERIMENT['PLANNER_BACKEND']),
    planner_surprise_name=str(EXPERIMENT['METRICS'][0]),
    method_labels=list(EXPERIMENT['METHODS']),
    resume_mode=str(EXPERIMENT['RUN_MODE']),
    warn_on_config_drift=bool(EXPERIMENT['WARN_ON_CONFIG_DRIFT']),
)

EXPERIMENT['RUN_TAG'] = str(run_context.run_tag)
EXPERIMENT['RUN_TAG_BASE'] = str(run_context.run_tag)
EXPERIMENT['RUN_NAME'] = str(EXPERIMENT['RUN_NAME']).strip() or str(run_context.run_tag)
EXPERIMENT['SHARD_ID'] = int(run_context.shard_id)
EXPERIMENT['RESUME_FROM_EXISTING'] = bool(run_context.cfg.resume_from_existing)

report_run_context(run_context, display_fn=display)

report_root = Path(EXPERIMENT['REPORT_PERSIST_ROOT']) / f"{EXPERIMENT['RUN_PREFIX']}_{EXPERIMENT['RUN_NAME']}"
EXPERIMENT['REPORT_OUTPUT_ROOT'] = str(report_root)
report_load = load_surprise_potential_report(str(report_root))

manifest = report_load.manifest if isinstance(report_load.manifest, dict) else {}
loaded_metrics = sorted(set(str(x) for x in manifest.get('metrics', []))) if isinstance(manifest.get('metrics', []), list) else []
loaded_families = sorted(set(str(x) for x in manifest.get('counterfactual_families', []))) if isinstance(manifest.get('counterfactual_families', []), list) else []
loaded_methods = sorted(set(str(x) for x in manifest.get('methods', []))) if isinstance(manifest.get('methods', []), list) else []
if (not loaded_metrics) and (not report_load.frames.metric_method_rollup_df.empty) and ('metric' in report_load.frames.metric_method_rollup_df.columns):
    loaded_metrics = sorted(set(report_load.frames.metric_method_rollup_df['metric'].astype(str).tolist()))
if (not loaded_families) and (not report_load.frames.metric_method_rollup_df.empty) and ('counterfactual_family' in report_load.frames.metric_method_rollup_df.columns):
    loaded_families = sorted(set(report_load.frames.metric_method_rollup_df['counterfactual_family'].astype(str).tolist()))
if (not loaded_methods) and (not report_load.frames.metric_method_rollup_df.empty) and ('method' in report_load.frames.metric_method_rollup_df.columns):
    loaded_methods = sorted(set(report_load.frames.metric_method_rollup_df['method'].astype(str).tolist()))
expected_metrics = sorted(set(str(x) for x in list(EXPERIMENT['METRICS'])))
expected_families = sorted(set(str(x) for x in list(EXPERIMENT['COUNTERFACTUAL_FAMILIES'])))
expected_methods = sorted(set(str(x) for x in list(EXPERIMENT['METHODS'])))

REPORT_COMPATIBLE_FOR_REUSE = bool(
    bool(report_load.is_usable)
    and loaded_metrics == expected_metrics
    and loaded_families == expected_families
    and loaded_methods == expected_methods
)

print('[report] output_root =', report_root)
print('[report] existing_report_found =', bool(report_load.exists))
print('[report] report_is_usable =', bool(report_load.is_usable))
print('[report] reusable_for_current_config =', bool(REPORT_COMPATIBLE_FOR_REUSE))
if bool(report_load.exists):
    print('[report] existing tables =', sorted(report_load.table_paths.keys()))
if bool(report_load.exists) and (not bool(report_load.is_usable)):
    print('[report] integrity issues =', list(report_load.integrity_issues))


## Step 6 - Fast-Fail Validation Cell

Run a cheap probe/preflight/gate pass (main loop forced off). Abort early if diagnostics fail.


In [ ]:
FAST_FAIL_READY = False
FAST_FAIL_SKIPPED_FOR_REPORT_REUSE = bool(
    bool(EXPERIMENT['AUTO_LOAD_EXISTING_REPORT_IF_AVAILABLE'])
    and bool(REPORT_COMPATIBLE_FOR_REUSE)
    and (not bool(EXPERIMENT['FORCE_RERUN_MAIN_SWEEP']))
)

if FAST_FAIL_SKIPPED_FOR_REPORT_REUSE:
    FAST_FAIL_READY = True
    print('[fast-fail] skipped: reusable persisted report already loaded.')
else:
    fast_fail_metric = str(EXPERIMENT['METRICS'][0])
    fast_fail_family = str(EXPERIMENT['COUNTERFACTUAL_FAMILIES'][0])
    fast_fail_run_tag = f"{EXPERIMENT['RUN_TAG_BASE']}_fastfail_{fast_fail_family}_{fast_fail_metric}"

    fast_fail_bundle = run_surprise_potential_single_metric(
        metric=fast_fail_metric,
        run_tag=fast_fail_run_tag,
        persist_root=str(EXPERIMENT['PERSIST_ROOT']),
        counterfactual_family=fast_fail_family,
        n_shards=1,
        shard_id=0,
        run_mode='fresh',
        planner_backend=str(EXPERIMENT['PLANNER_BACKEND']),
        auto_run_main_loop_when_ready=False,
        run_main_loop_override=False,
        run_tag_prefix=str(EXPERIMENT['RUN_TAG_PREFIX']),
        method_labels=list(EXPERIMENT['METHODS']),
        n_eval_scenarios=int(EXPERIMENT['FAST_FAIL_N_EVAL_SCENARIOS']),
        strict_min_eval=int(EXPERIMENT['FAST_FAIL_STRICT_MIN_EVAL']),
        n_total_scenarios_floor=int(EXPERIMENT['FAST_FAIL_N_TOTAL_SCENARIOS_FLOOR']),
        quick_probe_enabled=True,
        surprise_gate_enabled=True,
        restore_from_upload=bool(EXPERIMENT['RESTORE_FROM_UPLOAD']),
        quick_probe_settings=dict(EXPERIMENT['QUICK_PROBE_SETTINGS']),
        stop_on_preflight_fail=False,
        stop_on_gate_fail=False,
        allow_main_loop_when_gate_fails=False,
    )

    FAST_FAIL_READY = bool(
        fast_fail_bundle.preflight_bundle.ready_for_main_loop
        and fast_fail_bundle.gate_bundle.ready_for_optimization
    )

    print('[fast-fail] ready_for_main_loop =', fast_fail_bundle.preflight_bundle.ready_for_main_loop)
    print('[fast-fail] ready_for_optimization =', fast_fail_bundle.gate_bundle.ready_for_optimization)
    print('[fast-fail] passed =', FAST_FAIL_READY)

    display(fast_fail_bundle.summary_df)

    if not FAST_FAIL_READY:
        raise RuntimeError('Fast-fail validation did not pass; stop before full sweep.')


## Step 7 - Main Execution Cell

Run the full metric sweep only when `RUN_ENABLED=True` and fast-fail passed.


In [ ]:
sweep_bundle = None

if bool(EXPERIMENT['AUTO_LOAD_EXISTING_REPORT_IF_AVAILABLE']) and bool(REPORT_COMPATIBLE_FOR_REUSE) and (not bool(EXPERIMENT['FORCE_RERUN_MAIN_SWEEP'])):
    print('[main] skipped: existing persisted report found; using loaded report artifacts.')
elif not bool(EXPERIMENT['RUN_ENABLED']):
    print('[main] skipped: RUN_ENABLED=False')
elif not bool(FAST_FAIL_READY):
    print('[main] skipped: FAST_FAIL_READY=False')
else:
    run_mode = 'fresh'
    if bool(EXPERIMENT['RESUME_FROM_EXISTING']):
        run_mode = str(EXPERIMENT['RUN_MODE'])

    sweep_bundle = run_surprise_potential_metric_sweep(
        metrics=list(EXPERIMENT['METRICS']),
        counterfactual_families=list(EXPERIMENT['COUNTERFACTUAL_FAMILIES']),
        persist_root=str(EXPERIMENT['PERSIST_ROOT']),
        run_tag_base=str(EXPERIMENT['RUN_TAG_BASE']),
        run_tag_prefix=str(EXPERIMENT['RUN_TAG_PREFIX']),
        method_labels=list(EXPERIMENT['METHODS']),
        n_shards=int(EXPERIMENT['N_SHARDS']),
        shard_id=EXPERIMENT['SHARD_ID'],
        run_mode=str(run_mode),
        planner_backend=str(EXPERIMENT['PLANNER_BACKEND']),
        continue_on_error=bool(EXPERIMENT['CONTINUE_ON_ERROR']),
        auto_run_main_loop_when_ready=bool(EXPERIMENT['AUTO_RUN_MAIN_LOOP_WHEN_READY']),
        run_main_loop_override=EXPERIMENT['RUN_MAIN_LOOP_OVERRIDE'],
        quick_probe_settings=dict(EXPERIMENT['QUICK_PROBE_SETTINGS']),
        n_eval_scenarios=int(EXPERIMENT['N_EVAL_SCENARIOS']),
        strict_min_eval=int(EXPERIMENT['STRICT_MIN_EVAL']),
        n_total_scenarios_floor=int(EXPERIMENT['N_TOTAL_SCENARIOS_FLOOR']),
        **dict(EXPERIMENT['SWEEP_KWARGS']),
    )
    report_surprise_potential_sweep(sweep_bundle, display_fn=display, preview_rows=40)


## Step 8 - Evaluation/Reporting Cell

Inspect sweep-level outcomes and per-run signal summaries.


In [ ]:
report_frames = None
rigorous_eval_frames = None

if sweep_bundle is None:
    if bool(report_load.is_usable):
        print('[report] showing loaded persisted report')
        if not report_load.frames.metric_method_rollup_df.empty:
            display(report_load.frames.metric_method_rollup_df)
        if not report_load.frames.metric_rank_df.empty:
            display(report_load.frames.metric_rank_df)
        if not report_load.rigorous_frames.paired_delta_df.empty:
            print('[report] rigorous paired delta (loaded)')
            display(report_load.rigorous_frames.paired_delta_df.head(200))
        if not report_load.rigorous_frames.rigorous_rank_df.empty:
            print('[report] rigorous rank (loaded)')
            display(report_load.rigorous_frames.rigorous_rank_df)
    else:
        print('[report] no sweep bundle and no usable persisted report found.')
else:
    summary_df = sweep_bundle.summary_df
    method_rollup_df = sweep_bundle.method_rollup_df
    artifacts_df = sweep_bundle.artifacts_df
    errors_df = sweep_bundle.errors_df

    display(summary_df)
    display(method_rollup_df)
    display(errors_df)
    display(artifacts_df.head(50))

    report_frames = build_surprise_potential_report_frames(sweep_bundle)
    if not report_frames.metric_method_rollup_df.empty:
        print('[report] metric x method rollup')
        display(report_frames.metric_method_rollup_df)
    if not report_frames.metric_rank_df.empty:
        print('[report] metric ranking (composite gains)')
        display(report_frames.metric_rank_df)

    rigorous_eval_frames = build_surprise_potential_rigorous_eval(
        artifact_index_df=sweep_bundle.artifacts_df,
        summary_df=sweep_bundle.summary_df,
        n_bootstrap=int(EXPERIMENT['RIGOROUS_BOOTSTRAP_SAMPLES']),
        random_seed=int(EXPERIMENT['RIGOROUS_RANDOM_SEED']),
    )
    if not rigorous_eval_frames.paired_delta_df.empty:
        print('[report] rigorous paired delta (joint vs baselines with bootstrap CIs)')
        display(rigorous_eval_frames.paired_delta_df.head(300))
    if not rigorous_eval_frames.metric_health_df.empty:
        print('[report] metric health')
        display(rigorous_eval_frames.metric_health_df)
    if not rigorous_eval_frames.rigorous_rank_df.empty:
        print('[report] rigorous rank')
        display(rigorous_eval_frames.rigorous_rank_df)

    for key, run_bundle in sweep_bundle.metric_runs.items():
        print(
            f"\n[run] {key} family={run_bundle.counterfactual_family} "
            f"metric={run_bundle.metric} run_tag={run_bundle.run_tag}"
        )
        if not run_bundle.signal_bundle.signal_summary_df.empty:
            display(run_bundle.signal_bundle.signal_summary_df)


## Step 9 - Export Cell

Persist notebook-level sweep summaries and an execution manifest under persistent storage.


In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
from IPython.display import Image as IPyImage

report_root = Path(EXPERIMENT['REPORT_OUTPUT_ROOT'])
report_root.mkdir(parents=True, exist_ok=True)

written = []
report_export = None
source_mode = 'loaded'

if sweep_bundle is not None:
    source_mode = 'fresh_run'
    report_export = export_surprise_potential_report(
        bundle=sweep_bundle,
        output_root=str(report_root),
        write_plots=bool(EXPERIMENT['REPORT_WRITE_PLOTS']),
        write_rigorous_eval=True,
        rigorous_bootstrap=int(EXPERIMENT['RIGOROUS_BOOTSTRAP_SAMPLES']),
        rigorous_seed=int(EXPERIMENT['RIGOROUS_RANDOM_SEED']),
    )
    written.extend(sorted(report_export.table_paths.values()))
    written.extend(sorted(report_export.plot_paths.values()))
    written.append(str(report_export.manifest_path))
else:
    # Rehydrate from persisted report artifacts when the run already exists.
    if bool(report_load.is_usable):
        written.extend(sorted(report_load.table_paths.values()))
        written.extend(sorted(report_load.plot_paths.values()))
        written.append(str(Path(report_root) / 'report_manifest.json'))

notebook_manifest = {
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'source_mode': str(source_mode),
    'run_name': str(EXPERIMENT['RUN_NAME']),
    'run_prefix': str(EXPERIMENT['RUN_PREFIX']),
    'run_tag': str(EXPERIMENT['RUN_TAG']),
    'run_tag_base': str(EXPERIMENT['RUN_TAG_BASE']),
    'persist_root': str(EXPERIMENT['PERSIST_ROOT']),
    'report_persist_root': str(EXPERIMENT['REPORT_PERSIST_ROOT']),
    'report_output_root': str(report_root),
    'auto_load_existing_report_if_available': bool(EXPERIMENT['AUTO_LOAD_EXISTING_REPORT_IF_AVAILABLE']),
    'force_rerun_main_sweep': bool(EXPERIMENT['FORCE_RERUN_MAIN_SWEEP']),
    'n_shards': int(EXPERIMENT['N_SHARDS']),
    'shard_id': int(EXPERIMENT['SHARD_ID']),
    'resume_from_existing': bool(EXPERIMENT['RESUME_FROM_EXISTING']),
    'run_enabled': bool(EXPERIMENT['RUN_ENABLED']),
    'fast_fail_ready': bool(FAST_FAIL_READY),
    'fast_fail_skipped_for_report_reuse': bool(FAST_FAIL_SKIPPED_FOR_REPORT_REUSE),
    'report_reusable_for_current_config': bool(REPORT_COMPATIBLE_FOR_REUSE),
    'report_load_is_usable': bool(report_load.is_usable),
    'report_integrity_issues': list(report_load.integrity_issues),
    'metrics': list(EXPERIMENT['METRICS']),
    'counterfactual_families': list(EXPERIMENT['COUNTERFACTUAL_FAMILIES']),
    'methods': list(EXPERIMENT['METHODS']),
    'rigorous_bootstrap_samples': int(EXPERIMENT['RIGOROUS_BOOTSTRAP_SAMPLES']),
    'rigorous_random_seed': int(EXPERIMENT['RIGOROUS_RANDOM_SEED']),
    'written_outputs': list(sorted(set(written))),
}
manifest_path = report_root / 'notebook_run_manifest.json'
manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding='utf-8')
written.append(str(manifest_path))

print('[export] report_root =', report_root)
print('[export] source_mode =', source_mode)
print('[export] files_written =', len(sorted(set(written))))
for path in sorted(set(written)):
    print(' -', path)

plot_paths = {}
if report_export is not None:
    plot_paths = dict(report_export.plot_paths)
elif bool(report_load.is_usable):
    plot_paths = dict(report_load.plot_paths)

if len(plot_paths) > 0:
    for key, path in sorted(plot_paths.items()):
        print(f'[plot] {key}: {path}')
        try:
            display(IPyImage(filename=path))
        except Exception as exc:
            print(f'[plot] display failed for {path}: {exc}')

if len(written) > 0:
    display(pd.DataFrame({'path': sorted(set(written))}))

print('[next] This notebook now supports run+resume-aware evaluation. A separate deep-dive evaluation notebook can be added later without rerunning simulation.')


## Notes
- Notebook cells orchestrate only.
- Core simulation/probe/gate/export logic stays in `src.workflows` and `src.closedloop`.
- Runtime/bootstrap concerns stay in `src.platform`.
